# Verificação do pré-processamento

**Tarefas:** T1.7–T1.10 · **Dono:** _(a definir)_

Bancada de trabalho: a lógica reutilizável fica em `src/`; aqui corre-se, explora-se e
guarda-se o que o `Projeto_Final.ipynb` vai ler (`reports/experiments.csv`, `reports/figures/`).
Regras da modelação: [docs/modelacao_base.md](../docs/modelacao_base.md).

Só **mostra e valida** o que o `src/` faz; não define regras novas (essas ficam em `docs/decisions.md`).

In [2]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de src/ (data/, reports/, models/) são relativos à raiz

import pandas as pd

from src.features.build import CATEGORICAL_FEATURES, DROPPED, FEATURES
from src.pipeline import build_preprocessor, cv_splits, load_model_data

data = load_model_data()

## 2.4 População e split por doente (D1–D7, D13)

In [3]:
pd.DataFrame({
    "encontros": [len(data.X_train), len(data.X_test)],
    "doentes": [data.groups_train.nunique(), None],
    "positivos": [data.y_train.mean(), data.y_test.mean()],
}, index=["treino", "teste"])

,encontros,doentes,positivos
treino,80041,56309.0,0.113392
teste,20011,NaN,0.113438


In [5]:
pd.DataFrame([{"fold": k, "n_val": len(va), "positivos_val": data.y_train.iloc[va].mean(),
               "doentes_partilhados": len(set(data.groups_train.iloc[tr]) & set(data.groups_train.iloc[va]))}
              for k, (tr, va) in enumerate(cv_splits(data))])

,fold,n_val,positivos_val,doentes_partilhados
0,0,16007,0.113325,0
1,1,16007,0.113388,0
2,2,16008,0.113381,0
3,3,16008,0.113381,0
4,4,16011,0.113484,0


## 2.5 Features do modelo (D14, D15)

In [4]:
print(len(FEATURES), "features;", len(DROPPED), "colunas excluídas")
pd.Series(DROPPED, name="motivo").to_frame()

34 features; 25 colunas excluídas


,motivo
weight,"96,9% missing, sem sinal"
examide,constante
citoglipton,constante
change,redundante com as colunas de medicamentos (P20)
diabetesMed,igual a n_meds >= 1 (P19)
diag_1,substituído por diag_1_group
diag_2,substituído por diag_2_group
diag_3,substituído por diag_3_group
age,substituído por age_band
admission_type_id,substituído por admission_type_id_desc (D5)


## 2.6 Pré-processamento (D16)

Ajustado só no treino. Aqui só para inspecionar as colunas geradas.

In [6]:
pre = build_preprocessor().fit(data.X_train)
names = pd.Series(pre.get_feature_names_out())
print(len(names), "colunas depois do pré-processamento")
names.str.split("_").str[0].value_counts().head(15)

160 colunas depois do pré-processamento


diag            30
medical         26
payer           15
admission       14
discharge       14
race             6
number           4
n                4
A1Cresult        4
max              4
metformin        4
glimepiride      4
glipizide        4
glyburide        4
pioglitazone     4
Name: count, dtype: int64